In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_13_try_0.pickle

trying: ['responses_df_2022_cell10', 'responses_df_2022']
me:  5
me:  5
trying: ['responses_df_2022_cell10', 'responses_df_2022']
me:  5
me:  5
trying: ['title_for_x_axis', 'title_for_x_axis_cell20', 'title_for_x_axis_cell22']
me:  6
me:  8
me:  10
trying: ['title_for_x_axis', 'title_for_x_axis_cell20', 'title_for_x_axis_cell22']
me:  6
me:  8
me:  10
trying: ['percentages_cell21']
me:  9
trying: ['load_survey_data']
me:  1
trying: ['question_of_interest_cell21']
me:  9
trying: ['factor']
me:  1
trying: ['directory']
me:  1
trying: ['file_name_2018']
me:  1
trying: ['percentages_cell19']
me:  7
trying: ['count_then_return_percent_19']
me:  7
trying: ['question_of_interest_cell19']
me:  7
trying: ['count_then_return_percent_23']
me:  11
trying: ['percentages_cell23']
me:  11
trying: ['question_of_interest_cell23']
me:  11
trying: ['orientation_for_chart']
me:  5
trying: ['country_col']
me:  12
trying: ['grouped']
me:  12
trying: ['bar_chart_multiple_choice_24']
me:  12
trying: ['india_p

me:  0
trying: ['responses_in_order']
me:  5
trying: ['question_of_interest']
me:  5
trying: ['sns']
me:  0
trying: ['pd']
me:  0
trying: ['glob']
me:  0
trying: ['base_dir_2022']
me:  1
trying: ['file_name_2019']
me:  1
trying: ['file_name_2022']
me:  1
trying: ['base_dir_2019']
me:  1
trying: ['base_dir_2018']
me:  1
trying: ['replace_hyphen_with_en_dash']
me:  2
trying: ['responses_df_2019']
me:  1
trying: ['responses_df_2018']
me:  1
trying: ['create_dataframe_of_counts_16']
me:  4
trying: ['percentages_per_country_df']
me:  4
trying: ['count_then_return_percent_17']
me:  5
trying: ['percentages_cell17']
me:  5
trying: ['question_name']
me:  6
trying: ['question_of_interest_cell18']
me:  6
trying: ['percentages']
me:  5
trying: ['subset_of_countries']
me:  6
trying: ['country_df_combined']
me:  6
trying: ['responses_df_2019_cell10']
me:  6
trying: ['question_name_alternate']
me:  6
trying: ['responses_df_2018_cell10']
me:  8
trying: ['question_name_alternate_cell18']
me:  6
trying:

me:  6
trying: ['add_year_column_to_dataframes_18']
me:  6
trying: ['responses_df_2020']
me:  6
trying: ['combine_subset_of_data_from_multiple_years_18']
me:  6
trying: ['age_df_combined_cell22']
me:  10
trying: ['country_df_combined_cell18']
me:  6
trying: ['gender_map']
me:  10
trying: ['combine_subset_of_data_from_multiple_years_22']
me:  10
trying: ['question_of_interest_cell22']
me:  10
trying: ['combine_subset_of_data_from_multiple_years_20']
me:  8
trying: ['count_then_return_percent_20']
me:  8
trying: ['title_for_x_axis_cell20']
me:  8
trying: ['add_year_column_to_dataframes_20']
me:  8
trying: ['question_of_interest_cell20']
me:  8
trying: ['age_df_combined']
me:  8
trying: ['grab_subset_of_data_25']
me:  13
trying: ['online_learning_platforms_df_2022']
me:  13
trying: ['question_of_interest_cell25']
me:  13
trying: ['base_dir_2021']
me:  1
trying: ['base_dir_2017']
me:  1
trying: ['file_name_2020']
me:  1
trying: ['file_name_2017']
me:  1
trying: ['directory_cell8']
me:  1
t

In [3]:
%%RecordEvent
%%time
### cell 14 ###

# Preprocess 2018 and 2019 datasets without modifying the originals
orig_q = 'On which platforms have you begun or completed data science courses'
alt_q = 'On which online platforms have you begun or completed data science courses'

col_map_2018 = {
    alt_q: orig_q,
    'Kaggle Learn': 'Kaggle Learn Courses',
    'Fast.AI': 'Fast.ai',
    'Online University Courses': 'University Courses (resulting in a university degree)'
}
val_map_2018 = {
    'Kaggle Learn': 'Kaggle Learn Courses',
    'Fast.AI': 'Fast.ai',
    'Online University Courses': 'University Courses (resulting in a university degree)'
}
col_map_2019 = {'Kaggle Courses (i.e. Kaggle Learn)': 'Kaggle Learn Courses'}
val_map_2019 = col_map_2019

# Rename only once per DataFrame
# 2018
_df = responses_df_2018_cell10.copy()
cols = _df.columns
for old, new in col_map_2018.items():
    cols = cols.str.replace(old, new, regex=False)
_df.columns = cols
# 2019
_df2 = responses_df_2019_cell10.copy()
cols2 = _df2.columns
for old, new in col_map_2019.items():
    cols2 = cols2.str.replace(old, new, regex=False)
_df2.columns = cols2

# Helper to grab, remap and clean only the question-specific columns
def grab_subset(df, question, vmap=None):
    # select columns containing the question
    mask = df.columns.str.contains(question)
    sub = df.loc[:, mask]
    if vmap:
        # restrict replace to just these columns
        sub = sub.replace(vmap)
    # extract labels after the last '-' and strip leading spaces
    labels = sub.columns.str.split('-', n=1).str[-1].str.lstrip()
    sub.columns = labels
    # drop respondents who didn't select any of these
    return sub.dropna(how='all', subset=labels)

# Combine and count per year
def combine_data(question, include_2017=False):
    years = ['2018','2019','2020','2021','2022']
    dfs   = [_df, _df2, responses_df_2020, responses_df_2021, responses_df_2022_cell10]
    vmaps = [val_map_2018, val_map_2019, None, None, None]
    if include_2017:
        years.insert(0, '2017')
        dfs.insert(0, responses_df_2017)
        vmaps.insert(0, None)
    parts = [
        grab_subset(df, question, vmap).assign(year=yr)
        for df, yr, vmap in zip(dfs, years, vmaps)
    ]
    combined = pd.concat(parts, ignore_index=True)
    counts = combined.groupby('year', as_index=False).count()
    return combined, counts

# Convert counts to percentages
def to_percentages(combined, counts):
    totals = combined['year'].value_counts()
    pct = counts.set_index('year').div(totals, axis=0) * 100
    return pct.reset_index()

# Run the pipeline
question_of_interest_cell26 = orig_q + '?'
learning_platform_df_combined, learning_platform_df_combined_counts = (
    combine_data(question_of_interest_cell26)
)
learning_platform_df_combined_percentages = to_percentages(
    learning_platform_df_combined,
    learning_platform_df_combined_counts
)
# Clean up column names
learning_platform_df_combined_percentages.columns = (
    learning_platform_df_combined_percentages.columns
    .str.replace('(resulting in a university degree)', '', regex=False)
)
# Select and melt only the desired platforms
keep_cols = [
    'year', 'Coursera', 'University Courses ', 'Kaggle Learn Courses',
    'Udemy', 'Udacity', 'DataCamp', 'edX', 'Fast.ai', 'None', 'Other'
]
subset = learning_platform_df_combined_percentages.loc[:, keep_cols]
value_vars = [c for c in subset.columns if c not in ('year', 'None', 'Other')]
df_cell26 = (
    subset
    .melt(id_vars='year', value_vars=value_vars)
    .sort_values(['year', 'value'])
    .rename(columns={'variable': ''})
)
df_cell26.info()

KeyError: "['Coursera', 'University Courses ', 'Kaggle Learn Courses', 'Udemy', 'Udacity', 'DataCamp', 'edX', 'Fast.ai', 'None', 'Other'] not in index"

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_14_try_1.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_14_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[14], f)


In [ ]:
opt_output = Out.get(4)